In [15]:
import utils
from email.message import EmailMessage

In [ ]:
list_res = utils.service.users().messages().list(userId='me', q='in:inbox', maxResults=1).execute()
messages = list_res.get('messages', [])

msg_id = messages[0]['id']
original_msg = service.users().messages().get(userId='me', id=msg_id, format='full').execute()

headers = original_msg.get('payload', {}).get('headers', [])
subject = next((h['value'] for h in headers if h['name'] == 'Subject'), 'No Subject')
snippet = original_msg.get('snippet', '')

def extract_body(payload):
    if 'parts' in payload:
        for part in payload['parts']:
            if part.get('mimeType') == 'text/plain' and 'data' in part.get('body', {}):
                return base64.urlsafe_bdecode(part['body']['data']).decode('utf-8')
    elif 'body' in payload and 'data' in payload['body']:
        return base64.urlsafe_bdecode(payload['body']['data']).decode('utf-8')
    return snippet

body_text = extract_body(original_msg.get('payload', {}))

new_message = EmailMessage()
new_message['To'] = ','.join(settings.email_list)
new_message['Subject'] = subject
new_message.set_content(body_text)

raw_bytes = new_message.as_bytes()
encoded_message = base64.urlsafe_bencode(raw_bytes).decode('utf-8')

sent_message = service.users().messages().send(
    userId='me',
    body={'raw': encoded_message}
).execute()

print(f"Successfully sent message ID: {sent_message['id']} to {target_recipient}")